<a href="https://colab.research.google.com/github/238trimethylnonane/idp-model/blob/main/protein_troubleshooting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1 — Setup + Drive Mount

%pip -q install biopython

import os
import re
import gzip
import glob
import json
import shutil
import html as html_lib
import numpy as np
import pandas as pd

from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from IPython.display import HTML, display
from google.colab import drive, output

drive.mount("/content/drive", force_remount=False)

print("Setup complete.")
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.
Drive mounted: True


In [ ]:
# Cell 2 — Define Paths + Find Data Files

PROJECT_DIR = "/content/drive/MyDrive/protein/protein_project"
RAW_FASTA_DIR = "/content/drive/MyDrive/protein/raw_fasta"

os.makedirs(PROJECT_DIR, exist_ok=True)

EXPECTED_PATHS = {
    "cancer_csv": os.path.join(PROJECT_DIR, "cancer-results-csv.csv"),
    "atp_csv": os.path.join(PROJECT_DIR, "atp-results-csv.csv"),
    "atp_fasta": os.path.join(RAW_FASTA_DIR, "atpbinding.fasta"),
    "iupred_output_1": "/content/master_iupred_results.txt",
    "iupred_output_2": os.path.join(PROJECT_DIR, "master_iupred_results.txt"),
}


def find_file(filename, search_root="/content/drive/MyDrive"):
    for root, dirs, files in os.walk(search_root):
        if filename in files:
            return os.path.join(root, filename)
    return None


# Fallback searches
if not os.path.exists(EXPECTED_PATHS["atp_fasta"]):
    found = find_file("atpbinding.fasta")
    if found:
        EXPECTED_PATHS["atp_fasta"] = found

disprot_fasta = find_file("DisProt release_2025_12 with_ambiguous_evidences.fasta")
disprot_fasta_alt = find_file("DisProt release_2025_12 with_ambiguous_evidences (1).fasta")

if disprot_fasta_alt:
    EXPECTED_PATHS["disprot_fasta"] = disprot_fasta_alt
elif disprot_fasta:
    EXPECTED_PATHS["disprot_fasta"] = disprot_fasta
else:
    EXPECTED_PATHS["disprot_fasta"] = None

uniprot_gz = find_file("uniprotkb_atp_binding_AND_proteins_with_2026_05_25.fasta.gz")
EXPECTED_PATHS["uniprot_atp_gz"] = uniprot_gz

print("Detected paths:")
for key, path in EXPECTED_PATHS.items():
    print(f"{key}: {path} | exists: {os.path.exists(path) if path else False}")

Detected paths:
cancer_csv: /content/drive/MyDrive/protein/protein_project/cancer-results-csv.csv | exists: True
atp_csv: /content/drive/MyDrive/protein/protein_project/atp-results-csv.csv | exists: True
atp_fasta: /content/drive/MyDrive/protein/protein_project/raw_fasta/atpbinding.fasta | exists: True
iupred_output_1: /content/master_iupred_results.txt | exists: False
iupred_output_2: /content/drive/MyDrive/protein/protein_project/master_iupred_results.txt | exists: False
disprot_fasta: /content/drive/MyDrive/DisProt release_2025_12 with_ambiguous_evidences (1).fasta | exists: True
uniprot_atp_gz: /content/drive/MyDrive/protein/protein_project/uniprotkb_atp_binding_AND_proteins_with_2026_05_25.fasta.gz | exists: True


In [ ]:
# Cell 3 — Load CSV + FASTA Data

def load_csv_if_exists(path, source_label):
    if path and os.path.exists(path):
        df = pd.read_csv(path)
        df["Dataset_Source"] = source_label
        print(f"Loaded {source_label}: {df.shape} from {path}")
        return df
    return pd.DataFrame()


def load_fasta_if_exists(path, source_label):
    if not path or not os.path.exists(path):
        return pd.DataFrame()

    records = []

    if path.endswith(".gz"):
        handle = gzip.open(path, "rt")
    else:
        handle = open(path, "r")

    with handle:
        for record in SeqIO.parse(handle, "fasta"):
            records.append({
                "ID": str(record.id),
                "Description": str(record.description),
                "Sequence": str(record.seq),
                "Dataset_Source": source_label
            })

    df = pd.DataFrame(records)
    print(f"Loaded {source_label} FASTA: {df.shape} from {path}")
    return df


loaded_frames = []

cancer_csv_df = load_csv_if_exists(EXPECTED_PATHS["cancer_csv"], "Cancer CSV")
atp_csv_df = load_csv_if_exists(EXPECTED_PATHS["atp_csv"], "ATP CSV")
atp_fasta_df = load_fasta_if_exists(EXPECTED_PATHS["atp_fasta"], "ATP FASTA")
disprot_fasta_df = load_fasta_if_exists(EXPECTED_PATHS["disprot_fasta"], "DisProt FASTA")
uniprot_atp_df = load_fasta_if_exists(EXPECTED_PATHS["uniprot_atp_gz"], "UniProt ATP FASTA")

for df in [cancer_csv_df, atp_csv_df, atp_fasta_df, disprot_fasta_df, uniprot_atp_df]:
    if not df.empty:
        loaded_frames.append(df)

if len(loaded_frames) == 0:
    raise FileNotFoundError("No usable CSV or FASTA files were found. Check your Google Drive paths.")

raw_data = pd.concat(loaded_frames, ignore_index=True, sort=False)

print("Combined raw data:", raw_data.shape)
print(raw_data.columns.tolist())
display(raw_data.head())

Loaded Cancer CSV: (10000, 28) from /content/drive/MyDrive/protein/protein_project/cancer-results-csv.csv
Loaded ATP CSV: (10000, 28) from /content/drive/MyDrive/protein/protein_project/atp-results-csv.csv
Loaded ATP FASTA FASTA: (35, 4) from /content/drive/MyDrive/protein/protein_project/raw_fasta/atpbinding.fasta
Loaded DisProt FASTA FASTA: (1879, 4) from /content/drive/MyDrive/DisProt release_2025_12 with_ambiguous_evidences (1).fasta
Loaded UniProt ATP FASTA FASTA: (853, 4) from /content/drive/MyDrive/protein/protein_project/uniprotkb_atp_binding_AND_proteins_with_2026_05_25.fasta.gz
Combined raw data: (22767, 31)
['modelEntityId', 'gene', 'isUniProtReferenceProteome', 'isUniProtReviewed', 'sequenceChecksum', 'sequenceVersionDate', 'uniprotAccession', 'uniprotId', 'uniprotDescription', 'taxId', 'organismScientificName', 'globalMetricValue', 'sequenceStart', 'sequenceEnd', 'sequence', 'modelCreatedDate', 'organismCommonNames', 'latestVersion', 'allVersions', 'organismScientificNameT

,modelEntityId,gene,isUniProtReferenceProteome,isUniProtReviewed,sequenceChecksum,sequenceVersionDate,uniprotAccession,uniprotId,uniprotDescription,taxId,...,geneSynonyms,proteinFullNames,proteinShortNames,complexPredictionAccuracy_ipSAE,complexPredictionAccuracy_pDockQ2,complexPredictionAccuracy_LIS,Dataset_Source,ID,Description,Sequence
0,AF-Q504U0-F1,C4orf46,True,True,NaN,NaN,Q504U0,NaN,Renal cancer differentiation gene 1 protein,9606.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Cancer CSV,NaN,NaN,NaN
1,AF-Q6WQI6-F1,HEPN1,True,True,NaN,NaN,Q6WQI6,NaN,Putative cancer susceptibility gene HEPN1 protein,9606.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Cancer CSV,NaN,NaN,NaN
2,AF-Q9GZY1-F1,PBOV1,True,True,NaN,NaN,Q9GZY1,NaN,Prostate and breast cancer overexpressed gene ...,9606.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Cancer CSV,NaN,NaN,NaN
3,AF-P38398-2-F1,BRCA1,True,True,NaN,NaN,P38398-2,NaN,Breast cancer type 1 susceptibility protein,9606.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Cancer CSV,NaN,NaN,NaN
4,AF-Q5PSV4-2-F1,BRMS1L,True,True,NaN,NaN,Q5PSV4-2,NaN,Breast cancer metastasis-suppressor 1-like pro...,9606.0,...,NaN,NaN,NaN,NaN,NaN,NaN,Cancer CSV,NaN,NaN,NaN


In [ ]:
# Cell 4 — Standardize Columns + Compute Biophysical Features

def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def clean_sequence(seq):
    seq = str(seq).upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)
    return seq


def find_col(df, possible_names):
    normalized_cols = {normalize_name(col): col for col in df.columns}

    for name in possible_names:
        key = normalize_name(name)
        if key in normalized_cols:
            return normalized_cols[key]

    for name in possible_names:
        key = normalize_name(name)
        for norm_col, real_col in normalized_cols.items():
            if key and key in norm_col:
                return real_col

    return None


def numeric_from_column(series):
    direct = pd.to_numeric(series, errors="coerce")

    if direct.notna().sum() > 0:
        return direct

    def parse_mean(value):
        nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", str(value))
        if not nums:
            return np.nan
        return float(np.mean([float(x) for x in nums]))

    return series.apply(parse_mean)


def compute_features_from_sequence(seq):
    seq = clean_sequence(seq)

    if len(seq) < 5:
        return {
            "Length": np.nan,
            "GRAVY": np.nan,
            "Isoelectric_Point": np.nan,
            "Net_Charge": np.nan
        }

    try:
        pa = ProteinAnalysis(seq)

        return {
            "Length": len(seq),
            "GRAVY": pa.gravy(),
            "Isoelectric_Point": pa.isoelectric_point(),
            "Net_Charge": pa.charge_at_pH(7.0)
        }

    except Exception:
        return {
            "Length": len(seq),
            "GRAVY": np.nan,
            "Isoelectric_Point": np.nan,
            "Net_Charge": np.nan
        }


id_col = find_col(raw_data, [
    "ID", "Entry", "Protein", "Protein ID", "Protein_ID",
    "modelEntityId", "UniProt", "Accession", "Gene", "Gene Name"
])

desc_col = find_col(raw_data, [
    "Description", "Protein Description", "proteinDescription",
    "Function", "Annotation", "Name", "Protein Name"
])

seq_col = find_col(raw_data, [
    "Sequence", "Protein Sequence", "AA Sequence",
    "Amino Acid Sequence", "fasta", "protein_sequence"
])

length_col = find_col(raw_data, [
    "Length", "Protein Length", "Sequence Length",
    "proteinLength", "aa_length", "sequence_length"
])

gravy_col = find_col(raw_data, [
    "GRAVY", "Hydrophobicity", "Hydrophobicity GRAVY",
    "Hydrophobicity (GRAVY)", "gravy_score", "avg_gravy", "mean_gravy"
])

disorder_col = find_col(raw_data, [
    "Mean_Disorder", "Mean Disorder", "Average_Disorder",
    "Average Disorder", "Disorder", "Disorder Score",
    "IUPred", "IUPred Score", "IUPred_Mean", "Mean IUPred",
    "IUPred2A", "Intrinsic Disorder", "Predicted Disorder"
])


features_df = pd.DataFrame()

features_df["ID"] = raw_data[id_col].astype(str) if id_col else [f"PROT_{i}" for i in range(len(raw_data))]
features_df["Description"] = raw_data[desc_col].astype(str) if desc_col else ""
features_df["Dataset_Source"] = raw_data["Dataset_Source"].astype(str) if "Dataset_Source" in raw_data.columns else "Unknown"

if seq_col:
    features_df["Sequence"] = raw_data[seq_col].astype(str).apply(clean_sequence)
else:
    features_df["Sequence"] = ""

# Compute sequence-derived features
computed = features_df["Sequence"].apply(compute_features_from_sequence).apply(pd.Series)

features_df["Length"] = computed["Length"]
features_df["GRAVY"] = computed["GRAVY"]
features_df["Isoelectric_Point"] = computed["Isoelectric_Point"]
features_df["Net_Charge"] = computed["Net_Charge"]

# Prefer existing numeric columns if present
if length_col:
    existing_length = numeric_from_column(raw_data[length_col])
    features_df["Length"] = features_df["Length"].fillna(existing_length)

if gravy_col:
    existing_gravy = numeric_from_column(raw_data[gravy_col])
    features_df["GRAVY"] = features_df["GRAVY"].fillna(existing_gravy)

if disorder_col:
    features_df["Mean_Disorder"] = numeric_from_column(raw_data[disorder_col])
else:
    features_df["Mean_Disorder"] = np.nan

# Normalize disorder if values look like percentages
valid_disorder = features_df["Mean_Disorder"].dropna()
if len(valid_disorder) > 0 and valid_disorder.max() > 1.5 and valid_disorder.max() <= 100:
    features_df["Mean_Disorder"] = features_df["Mean_Disorder"] / 100.0

features_df["Length"] = pd.to_numeric(features_df["Length"], errors="coerce")
features_df["GRAVY"] = pd.to_numeric(features_df["GRAVY"], errors="coerce")
features_df["Mean_Disorder"] = pd.to_numeric(features_df["Mean_Disorder"], errors="coerce")

print("Detected columns:")
print("ID:", id_col)
print("Description:", desc_col)
print("Sequence:", seq_col)
print("Length:", length_col)
print("GRAVY:", gravy_col)
print("Disorder:", disorder_col)

print("Feature dataframe:", features_df.shape)
display(features_df.head())

Detected columns:
ID: ID
Description: Description
Sequence: Sequence
Length: None
GRAVY: None
Disorder: None
Feature dataframe: (22767, 9)


,ID,Description,Dataset_Source,Sequence,Length,GRAVY,Isoelectric_Point,Net_Charge,Mean_Disorder
0,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN
1,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN
2,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN
3,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN
4,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Cell 5 — Stratify Proteins

atp_keywords = [
    "atp", "atp-binding", "atp binding", "atp-dependent",
    "atpase", "kinase", "helicase", "phosphotransferase",
    "synthase", "nucleotide-binding", "nucleotide binding"
]

cancer_keywords = [
    "cancer", "tumor", "tumour", "oncogene", "carcinoma",
    "malignant", "leukemia", "lymphoma", "metastasis",
    "oncoprotein", "suppressor", "apoptosis", "p53",
    "sarcoma", "glioblastoma", "ras", "myc", "mtor",
    "egfr", "braf", "angiogenesis", "neoplasm",
    "adenocarcinoma", "melanoma", "glioma", "biomarker",
    "proto-oncogene", "oncogenic"
]


def has_any_keyword(text, keywords):
    text = str(text).lower()
    return any(k in text for k in keywords)


def infer_flags(row):
    text = " ".join([
        str(row.get("ID", "")),
        str(row.get("Description", "")),
        str(row.get("Dataset_Source", ""))
    ]).lower()

    source = str(row.get("Dataset_Source", "")).lower()

    atp_flag = has_any_keyword(text, atp_keywords) or "atp" in source
    cancer_flag = has_any_keyword(text, cancer_keywords) or "cancer" in source

    return pd.Series({
        "ATP_Flag": bool(atp_flag),
        "Cancer_Flag": bool(cancer_flag)
    })


flags = features_df.apply(infer_flags, axis=1)
features_df = pd.concat([features_df, flags], axis=1)


def assign_stratification(row):
    if row["ATP_Flag"] and row["Cancer_Flag"]:
        return "Group D - Cancer + ATP Binding"
    elif row["Cancer_Flag"]:
        return "Cancer Associated"
    elif row["ATP_Flag"]:
        return "ATP Binding"
    else:
        return "Uncategorized"


features_df["Stratification"] = features_df.apply(assign_stratification, axis=1)
features_df["Functional_Group"] = features_df["Stratification"]

print("Stratification counts:")
print(features_df["Stratification"].value_counts())
display(features_df.head())

Stratification counts:
Stratification
ATP Binding                       10702
Cancer Associated                 10000
Uncategorized                      1879
Group D - Cancer + ATP Binding      186
Name: count, dtype: int64


,ID,Description,Dataset_Source,Sequence,Length,GRAVY,Isoelectric_Point,Net_Charge,Mean_Disorder,ATP_Flag,Cancer_Flag,Stratification,Functional_Group
0,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN,False,True,Cancer Associated,Cancer Associated
1,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN,False,True,Cancer Associated,Cancer Associated
2,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN,False,True,Cancer Associated,Cancer Associated
3,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN,False,True,Cancer Associated,Cancer Associated
4,nan,nan,Cancer CSV,NAN,NaN,NaN,NaN,NaN,NaN,False,True,Cancer Associated,Cancer Associated


In [ ]:
# Cell 6 — Merge Duplicates + Fill Mean Disorder Robustly

import os
import re
import glob
import numpy as np
import pandas as pd

if "features_df" not in globals():
    raise RuntimeError("features_df does not exist. Run Cells 1–5 first.")

def normalize_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def estimate_disorder_from_sequence(seq):
    """
    Backup estimate only.
    This is NOT IUPred. It estimates disorder tendency from amino acid composition.
    """
    seq = str(seq).upper()
    seq = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "", seq)

    if len(seq) < 5:
        return np.nan

    disorder_promoting = set("PESQKRGD")
    order_promoting = set("WFYILVC")

    disorder_frac = sum(aa in disorder_promoting for aa in seq) / len(seq)
    order_frac = sum(aa in order_promoting for aa in seq) / len(seq)
    proline_frac = seq.count("P") / len(seq)
    glycine_frac = seq.count("G") / len(seq)
    charged_frac = sum(aa in "DEKR" for aa in seq) / len(seq)

    score = (
        0.18
        + 1.05 * disorder_frac
        + 0.35 * charged_frac
        + 0.25 * proline_frac
        + 0.15 * glycine_frac
        - 0.75 * order_frac
    )

    return float(np.clip(score, 0, 1))


def find_iupred_outputs():
    search_patterns = [
        "/content/master_iupred_results.txt",
        "/content/atpbinding_iupred2a_output.txt",
        "/content/disprot_iupred2a_output.txt",
        "/content/*iupred*.txt",
        "/content/*IUPred*.txt",
        "/content/*disorder*.txt",
        os.path.join(PROJECT_DIR, "master_iupred_results.txt"),
        os.path.join(PROJECT_DIR, "*iupred*.txt"),
        os.path.join(PROJECT_DIR, "*IUPred*.txt"),
        os.path.join(PROJECT_DIR, "*disorder*.txt"),
    ]

    files = []

    for pattern in search_patterns:
        files.extend(glob.glob(pattern))

    files = list(dict.fromkeys([f for f in files if os.path.exists(f) and os.path.getsize(f) > 0]))

    return files


def parse_iupred_output(file_path):
    """
    Robust parser for IUPred2A output.

    Accepts common rows like:
    1 M 0.123
    2 A 0.456

    Splits proteins on:
    - FASTA headers: >
    - IUPred headers: # IUPred2A
    """

    ordered_means = []
    current_scores = []

    def flush():
        nonlocal current_scores

        if len(current_scores) > 0:
            ordered_means.append(float(np.mean(current_scores)))

        current_scores = []

    with open(file_path, "r", errors="ignore") as f:
        for raw_line in f:
            line = raw_line.strip()

            if not line:
                continue

            if line.startswith(">"):
                flush()
                continue

            if line.startswith("# IUPred2A"):
                flush()
                continue

            if line.startswith("#"):
                continue

            parts = line.split()

            if len(parts) < 2:
                continue

            score = None

            # Most common IUPred format: position residue score
            if len(parts) >= 3:
                try:
                    candidate = float(parts[2])
                    if 0 <= candidate <= 1:
                        score = candidate
                except Exception:
                    pass

            # Fallback: find first valid 0–1 float after first column
            if score is None:
                for p in parts[1:]:
                    try:
                        candidate = float(p)
                        if 0 <= candidate <= 1:
                            score = candidate
                            break
                    except Exception:
                        pass

            if score is not None:
                current_scores.append(score)

    flush()

    return ordered_means


# ------------------------------------------------------------
# Merge duplicate IDs
# ------------------------------------------------------------

features_df["_ID_KEY"] = features_df["ID"].astype(str).apply(normalize_name)

merged_rows = []

for key, group in features_df.groupby("_ID_KEY", dropna=False):
    row = {}

    row["ID"] = group["ID"].dropna().astype(str).iloc[0]

    descriptions = group["Description"].dropna().astype(str).unique().tolist()
    row["Description"] = " | ".join(descriptions[:3])

    sources = group["Dataset_Source"].dropna().astype(str).unique().tolist()
    row["Dataset_Source"] = " + ".join(sources)

    seqs = group["Sequence"].dropna().astype(str)
    seqs = seqs[seqs.str.len() > 0]
    row["Sequence"] = seqs.iloc[0] if len(seqs) > 0 else ""

    row["Length"] = group["Length"].dropna().max() if group["Length"].notna().sum() > 0 else np.nan
    row["GRAVY"] = group["GRAVY"].dropna().mean() if group["GRAVY"].notna().sum() > 0 else np.nan
    row["Isoelectric_Point"] = group["Isoelectric_Point"].dropna().mean() if "Isoelectric_Point" in group.columns and group["Isoelectric_Point"].notna().sum() > 0 else np.nan
    row["Net_Charge"] = group["Net_Charge"].dropna().mean() if "Net_Charge" in group.columns and group["Net_Charge"].notna().sum() > 0 else np.nan

    if "Mean_Disorder" in group.columns and group["Mean_Disorder"].notna().sum() > 0:
        row["Mean_Disorder"] = group["Mean_Disorder"].dropna().mean()
    else:
        row["Mean_Disorder"] = np.nan

    row["ATP_Flag"] = bool(group["ATP_Flag"].max()) if "ATP_Flag" in group.columns else False
    row["Cancer_Flag"] = bool(group["Cancer_Flag"].max()) if "Cancer_Flag" in group.columns else False

    if row["ATP_Flag"] and row["Cancer_Flag"]:
        row["Stratification"] = "Group D - Cancer + ATP Binding"
    elif row["Cancer_Flag"]:
        row["Stratification"] = "Cancer Associated"
    elif row["ATP_Flag"]:
        row["Stratification"] = "ATP Binding"
    else:
        row["Stratification"] = "Uncategorized"

    row["Functional_Group"] = row["Stratification"]

    merged_rows.append(row)

master_results = pd.DataFrame(merged_rows)

master_results["Mean_Disorder"] = pd.to_numeric(master_results["Mean_Disorder"], errors="coerce")


# ------------------------------------------------------------
# Important fix: treat fake all-zero disorder as missing
# ------------------------------------------------------------

valid = master_results["Mean_Disorder"].dropna()

if len(valid) > 0 and valid.abs().sum() == 0:
    print("Mean_Disorder contains only 0.0 values. Treating as missing.")
    master_results["Mean_Disorder"] = np.nan


# ------------------------------------------------------------
# Fill from IUPred output files
# ------------------------------------------------------------

iupred_files = find_iupred_outputs()
print("IUPred output files found:")
for f in iupred_files:
    print(" -", f, "| size:", os.path.getsize(f), "bytes")

filled_from_iupred = 0

if len(iupred_files) > 0:
    for file_path in iupred_files:
        parsed_means = parse_iupred_output(file_path)

        print(f"Parsed {len(parsed_means)} protein-level disorder means from {file_path}")

        if len(parsed_means) == 0:
            continue

        missing_idx = master_results.index[master_results["Mean_Disorder"].isna()].tolist()
        n = min(len(missing_idx), len(parsed_means))

        if n > 0:
            master_results.loc[missing_idx[:n], "Mean_Disorder"] = parsed_means[:n]
            filled_from_iupred = n
            print(f"Filled {n} Mean_Disorder values from IUPred output.")
            break


# ------------------------------------------------------------
# Backup fill from sequence estimate if IUPred was unavailable
# ------------------------------------------------------------

if master_results["Mean_Disorder"].notna().sum() == 0:
    print("No usable IUPred disorder values found. Using sequence-based estimated disorder as backup.")
    master_results["Mean_Disorder"] = master_results["Sequence"].apply(estimate_disorder_from_sequence)
    master_results["Disorder_Source"] = "Sequence composition estimate"
else:
    master_results["Disorder_Source"] = "IUPred2A output"


# ------------------------------------------------------------
# Final cleanup + save
# ------------------------------------------------------------

master_results["Mean_Disorder"] = pd.to_numeric(master_results["Mean_Disorder"], errors="coerce")
master_results["GRAVY"] = pd.to_numeric(master_results["GRAVY"], errors="coerce")
master_results["Length"] = pd.to_numeric(master_results["Length"], errors="coerce")

output_path = os.path.join(PROJECT_DIR, "master_results_dashboard.csv")
master_results.to_csv(output_path, index=False)

print("\nFinal master_results:", master_results.shape)
print("Valid Mean_Disorder values:", master_results["Mean_Disorder"].notna().sum())
print("Mean disorder average:", master_results["Mean_Disorder"].dropna().mean())
print("Disorder source counts:")
print(master_results["Disorder_Source"].value_counts(dropna=False))

print("\nSaved:", output_path)

display(master_results[["ID", "GRAVY", "Mean_Disorder", "Disorder_Source", "Stratification"]].head(20))

IUPred output files found:
No usable IUPred disorder values found. Using sequence-based estimated disorder as backup.

Final master_results: (2762, 14)
Valid Mean_Disorder values: 2760
Mean disorder average: 0.6808109543742109
Disorder source counts:
Disorder_Source
Sequence composition estimate    2762
Name: count, dtype: int64

Saved: /content/drive/MyDrive/protein/protein_project/master_results_dashboard.csv


,ID,GRAVY,Mean_Disorder,Disorder_Source,Stratification
0,disprot|DP00012r005,-0.685484,0.679194,Sequence composition estimate,Uncategorized
1,disprot|DP00012r006,-0.685484,0.679194,Sequence composition estimate,Uncategorized
2,disprot|DP00012r007,-0.763243,0.682162,Sequence composition estimate,Uncategorized
3,disprot|DP00012r008,-0.763243,0.682162,Sequence composition estimate,Uncategorized
4,disprot|DP00012r010,-0.685484,0.679194,Sequence composition estimate,Uncategorized
5,disprot|DP00012r012,-0.561290,0.631613,Sequence composition estimate,Uncategorized
6,disprot|DP00012r013,-0.965000,0.592500,Sequence composition estimate,Uncategorized
7,disprot|DP00012r014,0.214286,0.462143,Sequence composition estimate,Uncategorized
8,disprot|DP00012r015,0.214286,0.462143,Sequence composition estimate,Uncategorized
9,disprot|DP00012r016,-0.965000,0.592500,Sequence composition estimate,Uncategorized


In [ ]:
# Cell 7 — Checkpoint Before Dashboard

required_cols = ["ID", "Length", "GRAVY", "Mean_Disorder", "Stratification"]

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("master_results exists:", "master_results" in globals())

if "master_results" in globals():
    print("Shape:", master_results.shape)
    print("Columns:", master_results.columns.tolist())

    for col in required_cols:
        print(f"{col} exists:", col in master_results.columns)

    print("Valid GRAVY values:", master_results["GRAVY"].notna().sum())
    print("Valid Mean_Disorder values:", master_results["Mean_Disorder"].notna().sum())
    print("Group counts:")
    print(master_results["Stratification"].value_counts())

    display(master_results[required_cols].head())
else:
    raise RuntimeError("master_results was not created. Re-run Cells 1–6.")

Drive mounted: True
master_results exists: True
Shape: (2762, 14)
Columns: ['ID', 'Description', 'Dataset_Source', 'Sequence', 'Length', 'GRAVY', 'Isoelectric_Point', 'Net_Charge', 'Mean_Disorder', 'ATP_Flag', 'Cancer_Flag', 'Stratification', 'Functional_Group', 'Disorder_Source']
ID exists: True
Length exists: True
GRAVY exists: True
Mean_Disorder exists: True
Stratification exists: True
Valid GRAVY values: 2760
Valid Mean_Disorder values: 2760
Group counts:
Stratification
Uncategorized                     1879
ATP Binding                        696
Group D - Cancer + ATP Binding     187
Name: count, dtype: int64


,ID,Length,GRAVY,Mean_Disorder,Stratification
0,disprot|DP00012r005,124.0,-0.685484,0.679194,Uncategorized
1,disprot|DP00012r006,124.0,-0.685484,0.679194,Uncategorized
2,disprot|DP00012r007,185.0,-0.763243,0.682162,Uncategorized
3,disprot|DP00012r008,185.0,-0.763243,0.682162,Uncategorized
4,disprot|DP00012r010,124.0,-0.685484,0.679194,Uncategorized


In [ ]:
# Cell 8 — Dashboard

import json
import html as html_lib
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from google.colab import output

if "master_results" not in globals():
    raise RuntimeError("master_results does not exist. Run Cells 1–7 first.")

df_dash = master_results.copy()

for col in ["ID", "Length", "GRAVY", "Mean_Disorder", "Stratification"]:
    if col not in df_dash.columns:
        df_dash[col] = np.nan

df_dash["ID"] = df_dash["ID"].astype(str)
df_dash["Length"] = pd.to_numeric(df_dash["Length"], errors="coerce")
df_dash["GRAVY"] = pd.to_numeric(df_dash["GRAVY"], errors="coerce")
df_dash["Mean_Disorder"] = pd.to_numeric(df_dash["Mean_Disorder"], errors="coerce")
df_dash["Stratification"] = df_dash["Stratification"].fillna("Uncategorized").astype(str)

avg_disorder = df_dash["Mean_Disorder"].dropna().mean()
avg_gravy = df_dash["GRAVY"].dropna().mean()

def fmt(x):
    if pd.isna(x):
        return "N/A"
    return str(round(float(x), 3))

kpis = {
    "total": len(df_dash),
    "avg_disorder": fmt(avg_disorder),
    "avg_gravy": fmt(avg_gravy),
    "group_d": int(df_dash["Stratification"].str.contains("Group D", case=False, na=False).sum())
}

scatter_df = df_dash[
    df_dash["GRAVY"].notna() &
    df_dash["Mean_Disorder"].notna()
][["ID", "GRAVY", "Mean_Disorder"]].head(1500)

group_summary = (
    df_dash.groupby("Stratification", dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

chart_data = scatter_df.replace([np.inf, -np.inf], np.nan).where(pd.notna(scatter_df), None).to_dict(orient="records")
group_data = group_summary.to_dict(orient="records")

source_note = (
    f"Proteins: {len(df_dash)} | "
    f"Valid GRAVY: {df_dash['GRAVY'].notna().sum()} | "
    f"Valid Mean Disorder: {df_dash['Mean_Disorder'].notna().sum()}"
)

def _report_js_error(message):
    print(f"Dashboard JS Error: {message}")

try:
    output.register_callback("report_js_error", _report_js_error)
except Exception:
    pass


html_code = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/ngl@2.0.0-dev.37/dist/ngl.js"></script>

    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f4f6f8;
            margin: 0;
            padding: 12px;
            color: #2d3436;
        }

        .top-panel {
            max-width: 980px;
            margin: 0 auto 12px auto;
            background: white;
            border-radius: 12px;
            padding: 14px 18px;
            box-shadow: 0 3px 8px rgba(0,0,0,0.06);
            display: grid;
            grid-template-columns: 300px 1fr;
            gap: 18px;
            box-sizing: border-box;
        }

        .label {
            font-size: 12px;
            font-weight: 700;
            color: #636e72;
            text-transform: uppercase;
            margin-bottom: 6px;
        }

        select {
            width: 100%;
            padding: 8px 10px;
            border-radius: 8px;
            border: 1px solid #dfe6e9;
            color: #0984e3;
            font-weight: 700;
            background: white;
        }

        .sync-good {
            color: #00b894;
            font-size: 14px;
            font-weight: 700;
            margin-bottom: 4px;
        }

        .source-note {
            color: #636e72;
            font-size: 11px;
        }

        .dashboard-container {
            max-width: 980px;
            margin: 0 auto;
            display: grid;
            grid-template-columns: repeat(4, 1fr);
            gap: 12px;
        }

        .card {
            background: white;
            border-radius: 12px;
            padding: 14px;
            box-shadow: 0 3px 8px rgba(0,0,0,0.06);
            box-sizing: border-box;
        }

        .kpi-card {
            height: 110px;
            text-align: center;
            display: flex;
            flex-direction: column;
            justify-content: center;
            border-top: 4px solid #0984e3;
        }

        .priority-card {
            border-top-color: #e17055;
        }

        .kpi-label {
            font-size: 11px;
            font-weight: 700;
            color: #636e72;
            text-transform: uppercase;
            margin-bottom: 8px;
        }

        .kpi-value {
            font-size: 30px;
            font-weight: 800;
            color: #0984e3;
        }

        .priority-card .kpi-value {
            color: #e17055;
        }

        .span-2 {
            grid-column: span 2;
        }

        .span-4 {
            grid-column: span 4;
        }

        h2 {
            font-size: 14px;
            margin: 0 0 10px 0;
        }

        .canvas-wrapper {
            height: 235px;
            position: relative;
        }

        #viewport-container {
            width: 100%;
            height: 330px;
            background: #2d3436;
            border-radius: 10px;
            position: relative;
            overflow: hidden;
        }

        #viewport {
            width: 100%;
            height: 100%;
        }

        #nglStatus {
            position: absolute;
            left: 10px;
            bottom: 10px;
            background: rgba(255,255,255,0.92);
            color: #2d3436;
            padding: 7px 10px;
            border-radius: 7px;
            font-size: 11px;
            font-weight: 700;
            z-index: 20;
        }

        .chart-warning {
            position: absolute;
            inset: 0;
            display: none;
            align-items: center;
            justify-content: center;
            text-align: center;
            color: #636e72;
            font-size: 12px;
            font-weight: 700;
            padding: 14px;
        }
    </style>
</head>

<body>
    <div class="top-panel">
        <div>
            <div class="label">3D Molecular Stage</div>
            <select id="pdbSelect">
                <option value="https://files.rcsb.org/download/3PQR.pdb" selected>Aurora Kinase A — 3PQR</option>
                <option value="https://files.rcsb.org/download/2SRC.pdb">SRC Kinase — 2SRC</option>
                <option value="https://files.rcsb.org/download/1OLG.pdb">p53 Tetramer — 1OLG</option>
                <option value="https://files.rcsb.org/download/4HHB.pdb">Hemoglobin — 4HHB</option>
                <option value="https://files.rcsb.org/download/1AID.pdb">HIV Protease — 1AID</option>
            </select>
        </div>

        <div>
            <div class="label">Synchronization State</div>
            <div class="sync-good">● Connected: master_results + charts + 3D model</div>
            <div class="source-note">__SOURCE_NOTE__</div>
        </div>
    </div>

    <div class="dashboard-container">
        <div class="card kpi-card">
            <div class="kpi-label">Protein Cohort</div>
            <div class="kpi-value">__TOTAL__</div>
        </div>

        <div class="card kpi-card">
            <div class="kpi-label">Mean Disorder</div>
            <div class="kpi-value">__DISORDER__</div>
        </div>

        <div class="card kpi-card">
            <div class="kpi-label">Mean GRAVY</div>
            <div class="kpi-value">__GRAVY__</div>
        </div>

        <div class="card kpi-card priority-card">
            <div class="kpi-label">Group D Identified</div>
            <div class="kpi-value">__GROUPD__</div>
        </div>

        <div class="card span-2">
            <h2>Functional Stratification Distribution</h2>
            <div class="canvas-wrapper">
                <canvas id="pieChart"></canvas>
            </div>
        </div>

        <div class="card span-2">
            <h2>Biophysical Correlation: Disorder vs GRAVY</h2>
            <div class="canvas-wrapper">
                <canvas id="scatterChart"></canvas>
                <div id="scatterWarning" class="chart-warning">
                    No valid GRAVY + Mean Disorder pairs were detected.
                </div>
            </div>
        </div>

        <div class="card span-4">
            <h2>3D Molecular Structural Stage</h2>
            <div id="viewport-container">
                <div id="viewport"></div>
                <div id="nglStatus">Loading structure...</div>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message, source, lineno, colno, error) {
            try {
                if (typeof google !== "undefined" && google.colab && google.colab.kernel) {
                    google.colab.kernel.invokeFunction("report_js_error", [String(message)], {});
                }
            } catch (e) {
                console.error(e);
            }
        };

        const rawData = __CHART_DATA__;
        const groupData = __GROUP_DATA__;

        const pieCanvas = document.getElementById("pieChart");

        if (pieCanvas && groupData.length > 0) {
            new Chart(pieCanvas, {
                type: "doughnut",
                data: {
                    labels: groupData.map(g => g.Stratification),
                    datasets: [{
                        data: groupData.map(g => Number(g.count)),
                        backgroundColor: [
                            "#0984e3", "#00b894", "#fdcb6e",
                            "#e17055", "#636e72", "#6c5ce7"
                        ]
                    }]
                },
                options: {
                    maintainAspectRatio: false,
                    plugins: {
                        legend: {
                            position: "bottom",
                            labels: { font: { size: 10 } }
                        }
                    }
                }
            });
        }

        const scatterCanvas = document.getElementById("scatterChart");
        const scatterWarning = document.getElementById("scatterWarning");

        if (scatterCanvas && rawData.length > 0) {
            new Chart(scatterCanvas, {
                type: "scatter",
                data: {
                    datasets: [{
                        data: rawData.map(d => ({
                            x: Number(d.GRAVY),
                            y: Number(d.Mean_Disorder)
                        })),
                        backgroundColor: "rgba(9,132,227,0.45)",
                        borderColor: "#0984e3",
                        pointRadius: 3
                    }]
                },
                options: {
                    maintainAspectRatio: false,
                    scales: {
                        x: { title: { display: true, text: "GRAVY" } },
                        y: {
                            title: { display: true, text: "Mean Disorder" },
                            suggestedMin: 0,
                            suggestedMax: 1
                        }
                    },
                    plugins: { legend: { display: false } }
                }
            });
        } else if (scatterWarning) {
            scatterWarning.style.display = "flex";
        }

        let stage = null;

        function setNglStatus(message) {
            const box = document.getElementById("nglStatus");
            if (box) box.textContent = message;
        }

        function initNGL() {
            if (typeof NGL === "undefined") {
                setNglStatus("Error: NGL failed to load.");
                return;
            }

            if (stage !== null) {
                return;
            }

            stage = new NGL.Stage("viewport", {
                backgroundColor: "#2d3436"
            });

            window.addEventListener("resize", function() {
                if (stage) stage.handleResize();
            });

            const selector = document.getElementById("pdbSelect");

            selector.addEventListener("change", function() {
                loadStructure(this.value);
            });

            loadStructure(selector.value);
        }

        function loadStructure(url) {
            if (!stage) {
                setNglStatus("Error: stage not initialized.");
                return;
            }

            setNglStatus("Loading structure...");
            stage.removeAllComponents();

            stage.loadFile(url, { ext: "pdb" })
                .then(function(component) {
                    component.addRepresentation("cartoon", {
                        color: "residueindex",
                        radiusScale: 0.7
                    });

                    component.addRepresentation("ball+stick", {
                        sele: "hetero and not water",
                        visible: true,
                        radiusScale: 0.4
                    });

                    setTimeout(function() {
                        stage.handleResize();
                        component.autoView();
                    }, 300);

                    setTimeout(function() {
                        stage.handleResize();
                        component.autoView();
                    }, 1000);

                    setNglStatus("Structure loaded.");
                })
                .catch(function(error) {
                    console.error(error);
                    setNglStatus("Error: structure failed to load.");
                });
        }

        document.addEventListener("DOMContentLoaded", function() {
            setTimeout(initNGL, 300);
        });

        setTimeout(initNGL, 1000);
    </script>
</body>
</html>
"""

final_html = (
    html_code
    .replace("__TOTAL__", str(kpis["total"]))
    .replace("__DISORDER__", str(kpis["avg_disorder"]))
    .replace("__GRAVY__", str(kpis["avg_gravy"]))
    .replace("__GROUPD__", str(kpis["group_d"]))
    .replace("__SOURCE_NOTE__", html_lib.escape(source_note))
    .replace("__CHART_DATA__", json.dumps(chart_data, ensure_ascii=False))
    .replace("__GROUP_DATA__", json.dumps(group_data, ensure_ascii=False))
)

display(HTML(final_html))